In [ ]:
# Machine-specific paths live in config.yaml next to this notebook (gitignored).
# Copy config.example.yaml -> config.yaml and fill in your paths.
from phoreal.config import load_config
cfg = load_config('config.yaml')

### Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

#Import general libs you'll need
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

# Import functions from main
from phoreal.agg_configs import ATL_AGG_CONFIG, ALS_AGG_CONFIG, ATL08_AGG_CONFIG

# Import functions from the utils
from phoreal.io import readers, processing
from phoreal import analysis
from phoreal import plotting as plotter
from phoreal.als.swath import create_als_swath, transform_als_swath
from phoreal.geo.datum_transforms import convert_3d_nad83_to_wgs84
from phoreal.geo.datum_transforms import get_geoid_height



### Identify files



In [ ]:
atl03_dir = cfg['austin_atl_dir']
atl03_name = 'ATL03_20250616013703_00052802_007_01.h5'
atl03_name = 'ATL03_20181215070448_11860106_007_01.h5'
atl03_file = os.path.join(atl03_dir, atl03_name)

atl08_dir = cfg['austin_atl_dir']
atl08_name = 'ATL08_20250616013703_00052802_007_01.h5'
atl08_name = 'ATL08_20181215070448_11860106_007_01.h5'
atl08_file = os.path.join(atl08_dir, atl08_name)

# Define the ground track of interest
gt = 'gt2r'

# Identify ATX AOI
extent_file = cfg['austin_extent_file']

# ALS Geoid File
als_geoid_file = cfg['als_geoid_file']

### Get photon rate dataframe


In [ ]:
# Get photon rate DF
df_ph = readers.read_photon_dataframe(atl03_file, gt, atl08_file)


In [ ]:
np.median(df_ph.solar_elevation)

In [ ]:
df_ph['crosstrack'] = 0 

In [ ]:
# Filter df_ph
extent_gdf = gpd.read_file(extent_file)
df_ph = processing.filter_df_by_extent(df_ph, extent_gdf.total_bounds)


### Plot output 

Check to make sure it plots correctly

In [ ]:
import matplotlib.pyplot as plt
plotter.plot_alongtrack(df_ph.alongtrack, df_ph.h_ph, df_ph.atl08_class,'Example Plot')

### Process ALS Swath

Load the ALS Swath and preprocess it 

In [ ]:
atl03_basename = os.path.basename(atl03_name).split('.h5')[0]
print(atl03_basename)

In [ ]:
# Load swath if available
als_outdir = cfg['als_swath_dir']
file_out_name = f"{os.path.basename(atl03_name).split('.h5')[0]}_{gt}"
als_outfile = os.path.join(als_outdir, f'als_{file_out_name}.pqt')                
if os.path.exists(als_outfile):
    als_swath = pd.read_parquet(als_outfile)
else:
    als_swath = create_als_swath(extent_gdf, df_ph)
    als_swath = transform_als_swath(als_swath, 
        'EPSG:32614', 
        source_geoid_file=als_geoid_file,
        target_geoid_file=None, 
        input_units='feet', 
        source_datum='nad83'
    )
    als_swath.to_parquet(als_outfile)

In [ ]:
from phoreal.plotting import plot_alongtrack_atl_als3
plot_alongtrack_atl_als3(df_ph, als_swath,'ALS vs ATL08')

### Perform Geolocation Offset

Fine tune the fit between the ICESat-2 and the ALS swath.

In [ ]:
%load_ext autoreload
%autoreload 2

# Call geolocation offset script
from phoreal.geolocation import create_interpolator


df_ph['crosstrack'] = 0


In [ ]:
als_surface_interpolator = create_interpolator(als_swath, grid_resolution=1)

In [ ]:
from phoreal.geolocation import calculate_mae_cost
from scipy.optimize import minimize, brute, fmin


search_grid = (slice(-10, 10, 0.5), slice(-10, 10, 0.5))

result = brute(
    calculate_mae_cost, 
    ranges=search_grid, 
    args=(df_ph.alongtrack[df_ph.atl08_class == 1], 
    df_ph.crosstrack[df_ph.atl08_class == 1], 
    df_ph.h_ph[df_ph.atl08_class == 1], als_surface_interpolator),
    finish=fmin, 
    full_output=True 
)

optimal_shift_at, optimal_shift_xt = result[0]
best_mae = result[1]

print(f"AT shift: {optimal_shift_at:.2f} m")
print(f"XT shift: {optimal_shift_xt:.2f} m")
print(f"Best MAE: {result[1]:.2f} m")

In [ ]:
from phoreal.geolocation import calculate_z_shift


optimal_shift_z = calculate_z_shift(df_ph.alongtrack[df_ph.atl08_class == 1], 
                                    df_ph.crosstrack[df_ph.atl08_class == 1],  
                                    df_ph.h_ph[df_ph.atl08_class == 1], 
                                    optimal_shift_at, 
                                    optimal_shift_xt , 
                                    als_surface_interpolator)



In [ ]:
# apply crosstrack and alongtrack shift to the ALS swath to make the math easier
als_swath['alongtrack'] = als_swath['alongtrack'] - optimal_shift_at
als_swath['crosstrack'] = als_swath['crosstrack'] - optimal_shift_xt




In [ ]:
als_swath = als_swath[als_swath.alongtrack > np.min(df_ph.alongtrack)]
als_swath = als_swath[als_swath.alongtrack < np.max(df_ph.alongtrack)]


In [ ]:
df_ph = df_ph[df_ph.alongtrack > np.min(als_swath.alongtrack)]
df_ph = df_ph[df_ph.alongtrack < np.max(als_swath.alongtrack)]

In [ ]:
import geopandas as gpd

# Define roughly the bounding box for Austin (Min Long, Min Lat, Max Long, Max Lat)
# austin_bbox = (-98.1, 30.1, -97.5, 30.5)

# Load ONLY the buildings within that bounding box from the local file
# Note: Geopandas can read directly from the zipped file!
# austin_buildings = gpd.read_file(file_path, bbox=austin_bbox)

# print(f"Loaded {len(austin_buildings)} buildings in the Austin area.")
austin_building_file = cfg['austin_buildings_file']
#austin_buildings.to_file( austin_building_file,driver='GPKG')

In [ ]:
austin_buildings = gpd.read_file(austin_building_file)
len(austin_buildings)

In [ ]:
%load_ext autoreload
%autoreload 2

from phoreal.als.swath import prepare_icesat2_track
from building_processing import find_intersected_buildings, convert_buildings_to_atxt, remove_complex_intersections

# 1. Prepare data once
gdf_buildings_utm = austin_buildings.to_crs('EPSG:32614')

is2_line_utm, line_x, line_y, line_at_dist = prepare_icesat2_track(
    df_ph, utm_epsg='EPSG:32614', resolution_m=10.0
)

# 2. Pipeline the data through the modular functions
candidates_utm = find_intersected_buildings(
    is2_line_utm, gdf_buildings_utm, buffer_meters=5.0, building_filter_size=300.0
)

# 3. Remove buildings that have complicated building shapes
clean_candidates = remove_complex_intersections(candidates_utm, is2_line_utm)


clean_candidates.to_file(f"austin_bldgs_clean_{atl03_basename}_{gt}.gpkg")


buildings_atxt = convert_buildings_to_atxt(
    clean_candidates, is2_line_utm, line_x, line_y, line_at_dist
)

In [ ]:
data = {
    'name': [f'{atl03_basename}_{gt}'],
    'geometry': [is2_line_utm]
}

# 3. Initialize the GeoDataFrame with a CRS
gdf = gpd.GeoDataFrame(data, crs="EPSG:32614")

# 4. Write to GeoPackage
# Use 'engine="pyogrio"' for high-performance writing
gdf.to_file(f"test_path2.gpkg", layer='layer',driver="GPKG", engine="pyogrio")

In [ ]:
candidates_utm.to_file('candidate_buildings.gpkg', driver='GPKG',index=True)

In [ ]:
%load_ext autoreload
%autoreload 2

from building_processing import filter_grazing_hits

hit_buildings = filter_grazing_hits(buildings_atxt)

In [ ]:
len(hit_buildings)

### Select individual building for analysis

In [ ]:
# target_building_id = 273324
# target_building_id = 17636
# target_building_id = 159174
# target_building_id = 2479
# target_building_id = 198116
# 12 is good
# 20 exit looks good
target_building_id = hit_buildings.iloc[9].name
target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
target_building_atxt = buildings_atxt.loc[target_building_id].geometry.buffer(20) 

In [ ]:
direction = 'exit'

In [ ]:
from building_processing import clip_als_to_buffered_building
target_als = clip_als_to_buffered_building(als_swath, target_building, buffer_m=1.0)
target_als = target_als[np.abs(target_als.crosstrack) < 6]

In [ ]:
cmap = plt.get_cmap('viridis')
# colors = cmap(target_als.h_norm[target_als.classification==6]) 
plt.scatter(target_als.crosstrack, target_als.alongtrack,c='k',label='Non-classified',s=3)
plt.scatter(target_als.crosstrack[target_als.classification==2], target_als.alongtrack[target_als.classification==2],c='orange',s=3,label='Ground')
plt.scatter(target_als.crosstrack[target_als.classification==3], target_als.alongtrack[target_als.classification==3],c='g',s=3,label='Vegetation')
plt.scatter(target_als.crosstrack[target_als.classification==6], target_als.alongtrack[target_als.classification==6],c='cyan',s=3,label='Building')
plt.plot(target_building_atxt.centroid.y, target_building_atxt.centroid.x,'o',label='Centroid')

plt.legend()
plt.title('Nadir view of target building by classification (at/xt)')
plt.xlabel('Crosstrack (m)')
plt.ylabel('Alongtrack (m)')
plt.axis('equal')
plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2
from building_processing import extract_building_edges_2d
edge = extract_building_edges_2d(target_als, target_building_atxt)
edge


In [ ]:
from building_processing import create_line
x_entry,y_entry = create_line(edge['entry']['slope'], edge['entry']['intercept'], x_start=-7, x_stop = 7)
x_exit,y_exit = create_line(edge['exit']['slope'], edge['exit']['intercept'], x_start=-7, x_stop = 7)

### Map View

In [ ]:
cmap = plt.get_cmap('viridis')
# colors = cmap(target_als.h_norm[target_als.classification==6]) 
plt.scatter(target_als.crosstrack, target_als.alongtrack,c='k',label='Non-classified',s=3)
plt.scatter(target_als.crosstrack[target_als.classification==2], target_als.alongtrack[target_als.classification==2],c='orange',s=3,label='Ground')
plt.scatter(target_als.crosstrack[target_als.classification==3], target_als.alongtrack[target_als.classification==3],c='g',s=3,label='Vegetation')
plt.scatter(target_als.crosstrack[target_als.classification==6], target_als.alongtrack[target_als.classification==6],c='cyan',s=3,label='Building')
plt.plot(x_entry,y_entry,'b--',label = 'Entry')
plt.plot(x_exit,y_exit,'r--',label = 'Exit')
plt.legend()
plt.title('Nadir view of target building by classification (at/xt)')
plt.xlabel('Crosstrack (m)')
plt.ylabel('Alongtrack (m)')
plt.axis('equal')
plt.show()

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.h_norm[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.h_norm[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.h_norm[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.h_norm[target_als.classification == 3],c='green',s=2,label='Vegetation')

plt.plot([edge['entry']['intercept'],edge['entry']['intercept']],[-5,20],'b--',label = 'Entry')
plt.plot([edge['exit']['intercept'],edge['exit']['intercept']],[-5,20],'r--',label = 'Exit')
plt.title('Profile view of target building')
plt.xlabel('Alongtrack (m)')
plt.ylabel('Normalized Height (m)')
plt.axis('equal')
plt.axis('tight')
plt.show()

In [ ]:
target_ph = df_ph[(edge[direction]['intercept'] + 100) > df_ph.alongtrack] 
target_ph = target_ph[(edge[direction]['intercept'] - 100) < df_ph.alongtrack] 
print(target_ph.columns)
# target_ph = target_ph[target_ph.h_norm < 130]
# target_ph = target_ph[target_ph.h_norm > -10]
len(target_ph)

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.h_norm[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.h_norm[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.h_norm[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.h_norm[target_als.classification == 3],c='green',s=2,label='Vegetation')
plt.scatter(target_ph.alongtrack,target_ph.h_norm,c='red',s=2,label='IS2')


plt.plot([edge['entry']['intercept'],edge['entry']['intercept']],[-5,20],'b--',label = 'Entry')
plt.plot([edge['exit']['intercept'],edge['exit']['intercept']],[-5,20],'r--',label = 'Exit')
plt.title('Profile view of target building')
plt.legend()
plt.xlabel('Alongtrack (m)')
plt.ylabel('Normalized Height (m)')
plt.axis('equal')
plt.show()

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.ellip_h[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.ellip_h[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.ellip_h[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.ellip_h[target_als.classification == 3],c='green',s=2,label='Vegetation')
plt.scatter(target_ph.alongtrack,target_ph.h_ph,c='red',s=2,label='IS2')

In [ ]:
%load_ext autoreload
%autoreload 2

from building_processing import calculate_orthogonal_distance, classify_photons
from building_processing import classify_photons_als_dynamic
target_ph['crosstrack'] = 0
target_ph = calculate_orthogonal_distance(target_ph, edge[direction], direction, threshold_m=30.0)

edge_als = calculate_orthogonal_distance(target_als, edge[direction], direction, threshold_m=15.0)

# target_ph = classify_photons(target_ph, edge[direction]['roof_median_h'], z_tolerance=1.0)
target_ph = classify_photons_als_dynamic(target_ph, edge_als, z_tolerance=1.0, fallback_tolerance=1.5)


In [ ]:
target_ph.attrs

### Orthometric Profile

In [ ]:
plt.plot(edge_als.dist_to_wall, 
edge_als.ellip_h,'.',color=[0.6,0.6,0.6], label='ALS Point')

# plt.plot(target_ph.dist_to_wall, 
# target_ph.h_ph,'.k',label='ICESat-2 Roof point')
plt.plot(target_ph.dist_to_wall[target_ph.target_class =='ground'], 
target_ph.h_ph[target_ph.target_class == 'ground'],'.r',label='ICESat-2 Ground Point')
plt.plot(target_ph.dist_to_wall[target_ph.target_class =='roof'], 
target_ph.h_ph[target_ph.target_class == 'roof'],'.b',label='ICESat-2 Roof point')

plt.legend(loc='center left')
plt.xlabel('Alongtrack (m)')
plt.ylabel('Elevation (m)')
plt.title('Example of Pre-Sight')
# plt.plot(target_ph.dist_to_wall[target_ph.target_class =='noise'], 
# target_ph.h_ph[target_ph.target_class == 'noise'],'.k')


# plt.plot(edge_als.dist_to_wall[edge_als.target_class =='roof'], 
# edge_als.h_norm[edge_als.target_class == 'roof'],'.b')
# plt.plot(edge_als.dist_to_wall[edge_als.target_class =='noise'], 
# edge_als.h_norm[edge_als.target_class == 'noise'],'.k')

In [ ]:
edge_als.columns

In [ ]:
ATL_AGG_CONFIG2 = [
    # Geometry
    {'field': 'x', 'operation': 'median', 'class_field': 'classification', 'class_id': list(range(1,100)), 'outfield': 'x_als'},
    {'field': 'y', 'operation': 'median', 'class_field': 'classification', 'class_id': list(range(1,100)), 'outfield': 'y_als'},

    # Terrain & Bathy
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [2], 'outfield': 'als_topo_median'},
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [40], 'outfield': 'als_bathy_median'},
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [41], 'outfield': 'als_surface_median'},

    # Terrain & Bathy
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [2], 'outfield': 'als_topo_ellip_median'},
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [40], 'outfield': 'als_bathy_ellip_median'},
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [41], 'outfield': 'als_surface_ellip_median'},


    # Canopy
    {'field': 'h_norm', 'operation': 'get_max98', 'class_field': 'classification', 'class_id': [3,4,5], 'outfield': 'als_norm_veg_max98'},
]

In [ ]:
df_seg = analysis.aggregate_by_segment(df_ph, ATL08_AGG_CONFIG, res=30)

In [ ]:
df_seg.columns

In [ ]:
als_beam = als_swath[np.abs(als_swath['crosstrack']) < (12/2)]
als_fwhm = als_swath[np.abs(als_swath['crosstrack']) < (6.56/2)]

In [ ]:
als_seg = analysis.aggregate_by_segment(als_beam, ALS_AGG_CONFIG, res=30)

In [ ]:
als_seg2 = analysis.aggregate_by_segment(als_fwhm, ALS_AGG_CONFIG, res=30)

In [ ]:
als_seg.columns

In [ ]:
df_seg.h_canopy[np.isnan(df_seg.h_canopy)] = 0
# df_seg.h_canopy[df_seg.h_canopy == 0] = np.nan

# als_seg.h_canopy[als_seg.als_norm_veg_max98 == 0] = np.nan
# als_seg2.h_canopy[als_seg2.als_norm_veg_max98 == 0] = np.nan

In [ ]:
err1_veg = als_seg.als_norm_veg_max98 - df_seg.h_canopy
print(np.abs(np.mean(err1_veg)))
print(np.sqrt(np.mean((err1_veg**2))))
print(np.mean(err1_veg))
print(len(err1_veg[~np.isnan(err1_veg)]))
# err1_terrain = als_seg.als_surface_ellip_median - df.h

In [ ]:
err2_veg = als_seg2.als_norm_veg_max98 - df_seg.h_canopy
print(np.abs(np.mean(err2_veg)))
print(np.sqrt(np.mean((err2_veg**2))))
print(np.mean(err2_veg))
print(len(err2_veg[~np.isnan(err2_veg)]))

In [ ]:
err1_ter = als_seg.als_topo_ellip_median - df_seg.h_te_median
print(np.abs(np.mean(err1_ter)))
print(np.sqrt(np.mean((err1_ter**2))))
print(np.mean(err1_ter))


In [ ]:
err2_ter = als_seg2.als_topo_ellip_median - df_seg.h_te_median
print(np.abs(np.mean(err2_ter)))
print(np.sqrt(np.mean((err2_ter**2))))
print(np.mean(err2_ter))


In [ ]:
plt.hist(err1_veg,range=[-10,10],bins=50,label='Error Residuals\nfor 11 m wide ALS Swath')
plt.hist(err2_veg,range=[-10,10],bins=50,alpha=0.5,label='Error Residuals\nfor 7.55 m wide ALS Swath')
plt.legend(loc = 'upper left')
plt.xlabel('Error (m)')
plt.ylabel('N Samples')
plt.title('Canopy Height Error')

In [ ]:
width_list = [12.83, 11, 10, 9, 8, 7, 6, 5, 4.9, 4, 3, 2, 1]
for width_test in width_list:
    als_fwhm = als_swath[np.abs(als_swath['crosstrack']) < (width_test/2)]
    als_seg2 = analysis.aggregate_by_segment(als_fwhm, ALS_AGG_CONFIG, res=30)
    err2_veg = als_seg2.als_norm_veg_max98 - df_seg.h_canopy
    mae = np.abs(np.mean(err2_veg))
    print(f'{width_test}: {mae}')


    


In [ ]:
# from building_processing import clip_als_to_buffered_building
# from building_processing import extract_building_edges_2d


# # target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
# for target_building_id in hit_buildings.index:
#     print(target_building_id)
#     target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
#     target_atxt = buildings_atxt.loc[target_building_id].geometry.buffer(20) 
#     target_als = clip_als_to_buffered_building(als_swath, target_building, buffer_m=1.0)
#     target_als = target_als[np.abs(target_als.crosstrack) < 7] 

#     edge = extract_building_edges_2d(target_als, target_atxt)
#     for direction in ['entry','exit']:
#         if edge[direction]['valid']:
            

#             out_name = f'{atl03_basename}_{gt}_{target_building_id}_{direction}'
#             print(out_name)

#             intercept = edge[direction]['intercept']
#             target_ph = df_ph[df_ph.alongtrack.between(intercept - 50, intercept + 50)]

#             from building_processing import calculate_orthogonal_distance, classify_photons
#             target_ph = calculate_orthogonal_distance(target_ph, edge[direction], direction, threshold_m=10.0)

#             target_ph = classify_photons(target_ph, edge[direction]['roof_median_h'], z_tolerance=1.0)

#             target_ph.to_parquet(f'outputs/{out_name}_out.pqt')


In [ ]:
df_ph.columns

In [ ]:
als_swath.columns

In [ ]:
import laspy
laspy.__version__

In [ ]:
def df_to_las(df, x_col, y_col, z_col, output_name, 
              intensity_col=None, classification_col=None, epsg=None):
    """
    Converts a pandas DataFrame to a LAS file using laspy 2.x.
    """
    
    # 1. Create a new header
    # Point format 3 is a good default as it supports GPS time and color, 
    # but point format 0 or 1 works for basic X, Y, Z, Intensity.
    header = laspy.LasHeader(point_format=3, version="1.2")

    # 2. Handle EPSG / CRS
    if epsg:
        header.add_crs(epsg)

    # 3. Create the LasData object
    las = laspy.LasData(header)

    # 4. Assign primary coordinates
    las.x = df[x_col].values
    las.y = df[y_col].values
    las.z = df[z_col].values

    # 5. Handle Optional Fields
    if intensity_col and intensity_col in df.columns:
        las.intensity = df[intensity_col].values
    
    if classification_col and classification_col in df.columns:
        las.classification = df[classification_col].values

    # 6. Write to file
    las.write(output_name)
    print(f"Successfully wrote {len(df)} points to {output_name}")

    return las

In [ ]:
df_ph.atl08_class

In [ ]:
out_file = os.path.join(cfg['las_export_dir'], f'df_{file_out_name}.las')
# df_to_las(df_ph.crosstrack, df_ph.alongtrack, df_ph.h_ph, out_file, 
#               intensity_col=None, classification_col=df_ph.atl08_class, epsg=None)

df_to_las(df_ph, 'crosstrack', 'alongtrack', 'h_ph', out_file, 
              intensity_col=None, classification_col='atl08_class', epsg=None)

In [ ]:
out_file = os.path.join(cfg['las_export_dir'], f'als1_{file_out_name}.las')
# df_to_las(df_ph.crosstrack, df_ph.alongtrack, df_ph.h_ph, out_file, 
#               intensity_col=None, classification_col=df_ph.atl08_class, epsg=None)

df_to_las(als_beam, 'crosstrack', 'alongtrack', 'ellip_h', out_file, 
              intensity_col='intensity', classification_col='classification', epsg=None)

In [ ]:
out_file = os.path.join(cfg['las_export_dir'], f'als2_{file_out_name}.las')
# df_to_las(df_ph.crosstrack, df_ph.alongtrack, df_ph.h_ph, out_file, 
#               intensity_col=None, classification_col=df_ph.atl08_class, epsg=None)

df_to_las(als_fwhm, 'crosstrack', 'alongtrack', 'ellip_h', out_file, 
              intensity_col='intensity', classification_col='classification', epsg=None)